# 🛒 Shopper Spectrum: Customer Segmentation & Product Recommendations

**Domain:** E-Commerce and Retail Analytics

This notebook covers:
1. Dataset loading & understanding
2. Data preprocessing
3. Exploratory Data Analysis (EDA)
4. RFM feature engineering
5. Clustering (KMeans) for customer segmentation
6. Item-based collaborative filtering for product recommendations
7. Saving the final model artifacts for the Streamlit app


## 1. Load & inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

df_raw = pd.read_csv("/mnt/user-data/uploads/online_retail.csv")
print(df_raw.shape)
df_raw.head()


In [ ]:
df_raw.info()
print()
print("Missing values:")
print(df_raw.isna().sum())
print()
print("Duplicate rows:", df_raw.duplicated().sum())


### Dataset columns

| Column | Description |
|---|---|
| InvoiceNo | Transaction number |
| StockCode | Unique product/item code |
| Description | Name of the product |
| Quantity | Number of products purchased |
| InvoiceDate | Date and time of transaction |
| UnitPrice | Price per product |
| CustomerID | Unique identifier for each customer |
| Country | Country where the customer is based |


## 2. Data preprocessing

In [ ]:
def load_and_clean(path):
    df = pd.read_csv(path)
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)

    # Drop missing CustomerID
    df = df.dropna(subset=["CustomerID"])

    # Exclude cancelled invoices (InvoiceNo starting with 'C')
    df = df[~df["InvoiceNo"].str.startswith("C")]

    # Remove negative or zero quantities and prices
    df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

    # Drop rows with missing description
    df = df.dropna(subset=["Description"])
    df["Description"] = df["Description"].str.strip()

    df["CustomerID"] = df["CustomerID"].astype(int)
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
    return df.reset_index(drop=True)

df = load_and_clean("/mnt/user-data/uploads/online_retail.csv")
print("Rows before cleaning:", len(df_raw), "| after cleaning:", len(df))
df.head()


## 3. Exploratory Data Analysis (EDA)

### Transaction volume by country

In [ ]:
country_counts = df["Country"].value_counts().head(10)
plt.figure(figsize=(9,5))
sns.barplot(x=country_counts.values, y=country_counts.index, palette="viridis")
plt.title("Top 10 Countries by Transaction Volume")
plt.xlabel("Number of transactions")
plt.show()


### Top-selling products

In [ ]:
top_products = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9,5))
sns.barplot(x=top_products.values, y=top_products.index, palette="magma")
plt.title("Top 10 Best-Selling Products by Quantity")
plt.xlabel("Total quantity sold")
plt.show()


### Purchase trends over time

In [ ]:
daily_sales = df.set_index("InvoiceDate").resample("D")["TotalPrice"].sum()
plt.figure(figsize=(12,5))
daily_sales.plot()
plt.title("Daily Revenue Over Time")
plt.ylabel("Revenue")
plt.show()


### Monetary distribution per transaction and per customer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.histplot(df["TotalPrice"], bins=60, ax=axes[0])
axes[0].set_xlim(0, df["TotalPrice"].quantile(0.99))
axes[0].set_title("Distribution of Transaction Line Value")

customer_spend = df.groupby("CustomerID")["TotalPrice"].sum()
sns.histplot(customer_spend, bins=60, ax=axes[1])
axes[1].set_xlim(0, customer_spend.quantile(0.99))
axes[1].set_title("Distribution of Total Spend per Customer")
plt.tight_layout()
plt.show()


## 4. RFM Feature Engineering

In [ ]:
snapshot_date = df["InvoiceDate"].max() + timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
).reset_index()

rfm.describe()


### RFM distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.histplot(rfm["Recency"], bins=40, ax=axes[0]).set_title("Recency")
sns.histplot(rfm["Frequency"], bins=40, ax=axes[1]).set_title("Frequency")
sns.histplot(rfm["Monetary"].clip(upper=rfm["Monetary"].quantile(0.99)), bins=40, ax=axes[2]).set_title("Monetary (99th pct clipped)")
plt.tight_layout()
plt.show()


## 5. Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rfm_log = rfm.copy()
for col in ["Recency", "Frequency", "Monetary"]:
    rfm_log[col] = np.log1p(rfm_log[col].clip(lower=0))

X = rfm_log[["Recency", "Frequency", "Monetary"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### Elbow curve & silhouette score for cluster selection

In [ ]:
inertias, sil_scores = [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    preds = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, preds))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_title("Elbow Curve (Inertia)")
axes[0].set_xlabel("k")
axes[1].plot(list(k_range), sil_scores, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

print(dict(zip(k_range, sil_scores)))


Silhouette score peaks at k=2, but a 2-cluster split is too coarse to be
actionable for marketing. We use **k=4** so the resulting segments map
cleanly onto the four business-defined archetypes (High-Value, Regular,
Occasional, At-Risk) requested in the brief.

In [ ]:
final_k = 4
kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)
rfm["Cluster"].value_counts()


### Customer cluster profiles

In [ ]:
profile = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean()
profile


In [ ]:
score = profile["Frequency"].rank() + profile["Monetary"].rank() - profile["Recency"].rank()
ordered = score.sort_values(ascending=False).index.tolist()
labels = ["High-Value", "Regular", "Occasional", "At-Risk"]
cluster_map = {cluster: labels[i] for i, cluster in enumerate(ordered)}
rfm["Segment"] = rfm["Cluster"].map(cluster_map)
print(cluster_map)
rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].mean()


### Visualize clusters (3D scatter of RFM)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection="3d")
colors = {"High-Value":"#2ecc71","Regular":"#3498db","Occasional":"#f39c12","At-Risk":"#e74c3c"}
for seg, group in rfm.groupby("Segment"):
    ax.scatter(group["Recency"], group["Frequency"], group["Monetary"],
               label=seg, alpha=0.5, color=colors.get(seg))
ax.set_xlabel("Recency")
ax.set_ylabel("Frequency")
ax.set_zlabel("Monetary")
ax.set_title("Customer Segments in RFM space")
ax.legend()
plt.show()


## 6. Item-Based Collaborative Filtering (Product Recommendations)

In [ ]:
basket = df.groupby(["CustomerID", "Description"])["Quantity"].sum().unstack(fill_value=0)
item_matrix = basket.T  # items x customers
print(item_matrix.shape)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(item_matrix)
sim_df = pd.DataFrame(sim, index=item_matrix.index, columns=item_matrix.index)
sim_df.shape


### Similarity heatmap (sample of top 15 best-selling products)

In [ ]:
top15 = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(15).index
plt.figure(figsize=(10,8))
sns.heatmap(sim_df.loc[top15, top15], cmap="viridis")
plt.title("Product Similarity Heatmap (Top 15 products)")
plt.show()


### Demo: recommend top-5 similar products

In [ ]:
def recommend(product_name, sim_df, top_n=5):
    matches = [p for p in sim_df.index if product_name.lower() in p.lower()]
    if not matches:
        return None, []
    match = matches[0]
    scores = sim_df[match].drop(labels=[match]).sort_values(ascending=False).head(top_n)
    return match, scores

match, recs = recommend("WHITE HANGING HEART T-LIGHT HOLDER", sim_df)
print("Matched product:", match)
recs


## 7. Model Evaluation Summary

In [ ]:
print("Final KMeans inertia:", kmeans.inertia_)
print("Final KMeans silhouette score:", silhouette_score(X_scaled, rfm['Cluster']))


## 8. Save artifacts for the Streamlit app

In [ ]:
import pickle, os

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

with open(os.path.join(MODEL_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(MODEL_DIR, "kmeans_model.pkl"), "wb") as f:
    pickle.dump(kmeans, f)
with open(os.path.join(MODEL_DIR, "cluster_labels.pkl"), "wb") as f:
    pickle.dump(cluster_map, f)
with open(os.path.join(MODEL_DIR, "similarity_matrix.pkl"), "wb") as f:
    pickle.dump(sim_df.astype("float32"), f)
with open(os.path.join(MODEL_DIR, "product_lookup.pkl"), "wb") as f:
    pickle.dump(df[["StockCode","Description"]].drop_duplicates("Description").set_index("Description")["StockCode"].to_dict(), f)
with open(os.path.join(MODEL_DIR, "rfm_table.pkl"), "wb") as f:
    pickle.dump(rfm, f)

print("Artifacts saved to", MODEL_DIR)
